In [1]:
# controle van mijn bestanden 

from pathlib import Path
import os

print("Beschikbare datasets:")
print(os.listdir("/kaggle/input"))

print("\nZoeken naar HMS-train.csv:")
for file in Path("/kaggle/input").rglob("train.csv"):
    print(file)

print("\nZoeken naar het baseline-script:")
for file in Path("/kaggle/input").rglob("hms_sklearn_baseline.py"):
    print(file)


Beschikbare datasets:
['competitions', 'datasets']

Zoeken naar HMS-train.csv:
/kaggle/input/competitions/hms-harmful-brain-activity-classification/train.csv

Zoeken naar het baseline-script:
/kaggle/input/datasets/hannaleysen/github-link-voor-knn/src/hms_sklearn_baseline.py


In [2]:
#bestaande baseline hergebruiken ! enkel de dingen veranderen die je meot veranderen
# Eerst de functies uit de baseline eventjes controleren

from pathlib import Path
import importlib.util
import inspect

# Python-Bestand zoeken in GitHub-dataset
scripts = list(
    Path("/kaggle/input").rglob("hms_sklearn_baseline.py")
)

if len(scripts) != 1:
    raise FileNotFoundError(
        f"Verwacht één baselinebestand, gevonden: {scripts}"
    )

SCRIPT_PATH = scripts[0]
print("Baseline gevonden op:", SCRIPT_PATH)

# Importeer bestaande baseline als Python-module
spec = importlib.util.spec_from_file_location(
    "hms_baseline",
    SCRIPT_PATH
)

baseline = importlib.util.module_from_spec(spec)
spec.loader.exec_module(baseline)

print("Baseline succesvol geïmporteerd!")
print("\nBeschikbare functies:")

for name in [
    "load_labels",
    "build_features",
    "kl_score",
    "cross_validate",
    "make_model",
    "to_matrix",
    "expand"
]:
    function = getattr(baseline, name, None)
    if function is not None:
        print(name, inspect.signature(function))


Baseline gevonden op: /kaggle/input/datasets/hannaleysen/github-link-voor-knn/src/hms_sklearn_baseline.py
Baseline succesvol geïmporteerd!

Beschikbare functies:
load_labels (data_dir, max_specs=None, seed=42)
build_features (meta, spec_dir, is_test=False, n_jobs=-1)
kl_score (y_true, y_pred, eps=1e-15)
cross_validate (X, Y, groups, num_votes, n_folds, seed)
make_model (name, seed=42)
to_matrix (frame, feats)
expand (X, Y)


In [3]:
# KNN --> de 6 expertkansen rechtstreeks gebruiken en niet eerst omzetten naar gewone klassevoorbeelden 

import inspect

print("Informatie over build_features:")
print(inspect.getsource(baseline.build_features))

print("\nInformatie over cross_validate:")
print(inspect.getsource(baseline.cross_validate))


Informatie over build_features:
def build_features(meta, spec_dir, is_test=False, n_jobs=-1):
    """Compute features for all EEGs in `meta`, reading each spectrogram file once (in parallel)."""
    t0 = time.time()
    if is_test:   # test.csv has no label offsets
        meta = meta.assign(off_min=0.0, off_max=0.0)
    jobs = [delayed(spectrogram_features)(spec_dir / f"{sid}.parquet",
                                          list(zip(g.eeg_id, g.off_min, g.off_max)), is_test)
            for sid, g in meta.groupby("spectrogram_id")]
    results = Parallel(n_jobs=n_jobs, batch_size=32)(jobs)
    out = pd.DataFrame([row for rows in results for row in rows])
    print(f"{len(jobs)} spectrograms -> {len(out)} feature rows in {time.time() - t0:.0f}s")
    return out


Informatie over cross_validate:
def cross_validate(X, Y, groups, num_votes, n_folds, seed):
    """Patient-grouped K-fold CV for both models; prints scores and returns the better model's name."""
    prior = np.tile(Y.mean(

In [4]:
# KNN-experiment 


In [5]:
# eerst nog dingetjes ophalen uit de baseline 
import inspect

print("MODEL_NAMES:")
print(baseline.MODEL_NAMES)              # welke modellen vergelijkt de baseline momenteel

print("\nFunctie fit:")
print(inspect.getsource(baseline.fit))   # Hoe worden bestaande modellen getraind 

print("\nFunctie load_labels:")
print(inspect.getsource(baseline.load_labels))  # hoe worden de 6 doelklassen ingelezen

print("\nFunctie to_matrix:")
print(inspect.getsource(baseline.to_matrix))    # om exact dezelfde featureselectie als de baseline te gebruiken

MODEL_NAMES:
['logreg', 'hgb']

Functie fit:
def fit(name, X, Y, seed=42):
    """Fit model `name` on features X and soft labels Y (rows are vote distributions)."""
    Xe, ye, we = expand(X, Y)
    m = make_model(name, seed)
    if name == "logreg":   # in a Pipeline the weight must be routed to the final step
        m.fit(Xe, ye, logisticregression__sample_weight=we)
    else:
        m.fit(Xe, ye, sample_weight=we)
    return m


Functie load_labels:
def load_labels(data_dir, max_specs=None, seed=42):
    """Return one row per EEG with its spectrogram id, label offsets, patient id and vote distribution."""
    train = pd.read_csv(data_dir / "train.csv")

    # train.csv has several labelled sub-segments per eeg_id: sum their votes and keep the offset range.
    agg = {"spectrogram_id": "first", "spectrogram_label_offset_seconds": ["min", "max"], "patient_id": "first"}
    agg.update({c: "sum" for c in CLASSES})
    df = train.groupby("eeg_id").agg(agg)
    df.columns = ["spectrogra

In [6]:
# Hoe brengt de baseline alles samen?  --> ervoor zorgen dat KNN dezelfde invoergegevens krijgt als de andere 2 modellen in de baseline 
import inspect

print("1. De functie main:")  
print(inspect.getsource(baseline.main))   # volgorde dataverwerking baseline (gehele workflow)

print("\n2. De functie make_model:")
print(inspect.getsource(baseline.make_model))   # hoe modellen instellen 

print("\n3. De beschikbare klassen:")
print(baseline.CLASSES)                   # volgorde doelklassen bevestigen (KNN moet deze ook hebben)

1. De functie main:
def main():
    parser = argparse.ArgumentParser(description="HMS simple scikit-learn baseline")
    parser.add_argument("--data-dir", type=Path, required=True,
                        help="folder that contains train.csv, test.csv, train_spectrograms/ and test_spectrograms/")
    parser.add_argument("--mode", choices=["full", "quick"], default="full",
                        help="full: all training spectrograms; quick: a random 3,000 for a fast check")
    parser.add_argument("--no-cv", action="store_true", help="skip cross-validation and use gradient boosting")
    parser.add_argument("--folds", type=int, default=5, help="number of cross-validation folds")
    parser.add_argument("--n-jobs", type=int, default=-1, help="CPU cores for reading spectrograms (-1 = all)")
    parser.add_argument("--output", type=Path, default=Path("submission.csv"), help="where to write predictions")
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()

Schalen is voor KNN zeer belangrijk --> berekent de afstanden tussen datapunten. Zonder schaling kunnen kenmerken met grote waarden de afstand beïnvloeden op een slechte manier 

Volgorde behouden: ['seizure_vote', 'lpd_vote', 'gpd_vote', 'lrda_vote', 'grda_vote', 'other_vote'] 

# KNN Code

In [7]:
# data inladen en features bouwen (zelfde gebruiken als baseline)
from pathlib import Path
import numpy as np
import pandas as pd

# Pad naar de HMS-competitiedataset
DATA_DIR = Path(
    "/kaggle/input/competitions/hms-harmful-brain-activity-classification"
)

# Controleer of de dataset gevonden wordt
assert (DATA_DIR / "train.csv").exists(), "train.csv niet gevonden!"

# 1. Labels inladen
df = baseline.load_labels(
    DATA_DIR,
    max_specs=3000,
    seed=42
)

print(df.shape, "EEGs")
print(df.patient_id.nunique(), "verschillende patiënten")
print(df.spectrogram_id.nunique(), "spectrogrammen")

# 2. Features berekenen
X_df = baseline.build_features(
    df,
    DATA_DIR / "train_spectrograms",
    n_jobs=2
)

# 3. Labels en features samenvoegen
df = df.merge(X_df, on="eeg_id")

# 4. Dezelfde featureselectie gebruiken als de baseline
feats = [c for c in X_df.columns if c != "eeg_id"]

X = baseline.to_matrix(df, feats)
Y = df[baseline.CLASSES].to_numpy(dtype=np.float64)

groups = df["patient_id"].to_numpy()
num_votes = df["num_votes"].to_numpy()

print("\nData klaar voor KNN!")
print("Aantal EEG-records:", len(X))
print("Aantal features:", X.shape[1])
print("Aantal klassen:", Y.shape[1])
print("Vorm van X:", X.shape)
print("Vorm van Y:", Y.shape)

(4738, 12) EEGs
1163 verschillende patiënten
3000 spectrogrammen
3000 spectrograms -> 4738 feature rows in 82s

Data klaar voor KNN!
Aantal EEG-records: 4738
Aantal features: 88
Aantal klassen: 6
Vorm van X: (4738, 88)
Vorm van Y: (4738, 6)


In [8]:
# KNN-model --> classifiër maken 
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

class SoftKNNClassifier:

    def __init__(self, n_neighbors=15):
        self.n_neighbors = n_neighbors
        self.scaler = StandardScaler()
        self.neighbors = None
        self.Y_train = None

    def fit(self, X, Y):

        # Schaal de features
        X_scaled = self.scaler.fit_transform(X)

        # Bewaar de zes kansen per trainingsrecord
        self.Y_train = np.asarray(Y, dtype=np.float64)

        # KNN-model trainen
        k = min(self.n_neighbors, len(X_scaled))
        self.neighbors = NearestNeighbors(n_neighbors=k, metric="euclidean")
        self.neighbors.fit(X_scaled)

        return self

    def predict_proba(self, X):

        X_scaled = self.scaler.transform(X)
        distances, indices = self.neighbors.kneighbors(X_scaled)

        # Dichtere buren krijgen meer gewicht
        weights = 1.0 / np.maximum(distances, 1e-8)

        # Als echt gelijk --> gebruik enkel die buren
        for i in range(len(distances)):
            exact = distances[i] < 1e-8
            if np.any(exact):
                weights[i] = exact.astype(float)

        # Haal de zes labelkansen van de buren op
        neighbor_labels = self.Y_train[indices]

        # Gewogen gemiddelde berekenen
        predictions = np.sum(
            neighbor_labels * weights[:, :, None],
            axis=1
        ) / weights.sum(axis=1, keepdims=True)

        # Normaliseren zodat de zes kansen samen 1 zijn
        predictions = np.maximum(predictions, 1e-12)
        predictions /= predictions.sum(axis=1, keepdims=True)

        return predictions


print("SoftKNNClassifier is aangemaakt!")

SoftKNNClassifier is aangemaakt!


In [9]:
# KNN trainen, testen en vergelijken van verschillende waarden voor k en vergelijken 
# --> gebruik zelfde patientgegevens als in baseline en vgl met de modellen in de baseline

from sklearn.model_selection import GroupKFold  # dataset opdelen in training- en testgegevens 

# Verschillende aantallen buren testen
K_VALUES = [5, 10, 15, 25, 40]   # welke waarde geeft beste KL-sore
N_FOLDS = 3
SEED = 42

# Resultaten bewaren
knn_oof = {
    k: np.zeros_like(Y)
    for k in K_VALUES}    # voor elke k-waarde een matrix aanmaken --> voorspellingen daarin bewaren 

baseline_oof = {
    name: np.zeros_like(Y)
    for name in baseline.MODEL_NAMES}

# Cross-validatie per patiënt
cv = GroupKFold(n_splits=N_FOLDS)  # data verdelen 

# elke groep afzonderlijk trainen en testen 
for fold, (tr, va) in enumerate(  
    cv.split(X, Y, groups)):
    print(f"\n--- Fold {fold + 1}/{N_FOLDS} ---")   
         # tr = rijen om model te trainen, va=rijden om model te testen
         # X = kenmerken, Y = 6 werkelijke labelkansen

    # Schaal de features uitsluitend op de trainingsdata
    scaler = StandardScaler()

    X_train = scaler.fit_transform(X[tr])
    X_valid = scaler.transform(X[va])

    # We berekenen het max # nodige buren, grootste k-waarde = 40
    max_k = min(max(K_VALUES), len(tr))

    neighbors = NearestNeighbors(
        n_neighbors=max_k,
        metric="euclidean"
    )

    neighbors.fit(X_train)

    distances, indices = neighbors.kneighbors(X_valid)

    # De labels van de trainingsburen ophalen
    neighbor_labels = Y[tr][indices]

    # Iedere k-waarde afzonderlijk evalueren 
    for k in K_VALUES:

        d = distances[:, :k]
        labels = neighbor_labels[:, :k, :]

        weights = 1.0 / np.maximum(d, 1e-8)

        # Exacte overeenkomsten voorrang geven
        exact = d < 1e-8

        rows_with_exact = exact.any(axis=1)
        weights[rows_with_exact] = exact[rows_with_exact].astype(float)

        predictions = np.sum(
            labels * weights[:, :, None],
            axis=1
        ) / weights.sum(axis=1, keepdims=True)

        predictions = np.maximum(predictions, 1e-12)
        predictions /= predictions.sum(axis=1, keepdims=True)

        knn_oof[k][va] = predictions

    # baseline modellen ook testen op zelfde trainings- & testdata
    for name in baseline.MODEL_NAMES:

        model = baseline.fit(
            name,
            X[tr],
            Y[tr],
            seed=SEED
        )

        baseline_oof[name][va] = model.predict_proba(X[va])

    # Tussentijdse scores tonen
    for k in K_VALUES:
        score = baseline.kl_score(Y[va], knn_oof[k][va])
        print(f"KNN k={k}: KL={score:.5f}")

    for name in baseline.MODEL_NAMES:
        score = baseline.kl_score(Y[va], baseline_oof[name][va])
        print(f"{name}: KL={score:.5f}")


# Definitieve scores berekenen --> KL-score over de volledige verzameling voorspellingen (OOF-voorspellingen)
results = []

for k in K_VALUES:
    score = baseline.kl_score(Y, knn_oof[k])
    results.append({
        "Model": f"KNN (k={k})",
        "KL-score": score
    })

for name in baseline.MODEL_NAMES:
    score = baseline.kl_score(Y, baseline_oof[name])
    results.append({
        "Model": name,
        "KL-score": score
    })

# Resultaten sorteren op KL-score en tonen (beste score bovenaan) 
results_df = pd.DataFrame(results)
results_df = results_df.sort_values("KL-score").reset_index(drop=True)

print("\n===== VERGELIJKING MODELLEN =====")
print(results_df.to_string(index=False))

# Beste model tonen 
print("\nBeste model:")
print(results_df.iloc[0])




--- Fold 1/3 ---
KNN k=5: KL=3.74621
KNN k=10: KL=2.05499
KNN k=15: KL=1.62042
KNN k=25: KL=1.20500
KNN k=40: KL=1.07394
logreg: KL=1.14147
hgb: KL=1.01937

--- Fold 2/3 ---
KNN k=5: KL=4.03807
KNN k=10: KL=2.43528
KNN k=15: KL=1.63300
KNN k=25: KL=1.32533
KNN k=40: KL=1.17630
logreg: KL=1.20710
hgb: KL=1.22767

--- Fold 3/3 ---
KNN k=5: KL=3.82406
KNN k=10: KL=2.03618
KNN k=15: KL=1.53308
KNN k=25: KL=1.17815
KNN k=40: KL=1.06244
logreg: KL=1.13272
hgb: KL=1.11826

===== VERGELIJKING MODELLEN =====
     Model  KL-score
KNN (k=40)  1.104220
       hgb  1.121746
    logreg  1.160423
KNN (k=25)  1.236155
KNN (k=15)  1.595505
KNN (k=10)  2.175459
 KNN (k=5)  3.869420

Beste model:
Model       KNN (k=40)
KL-score       1.10422
Name: 0, dtype: object
